# Quantitative Analysis — Milei Climate Narratives Corpus (v4)

Reproducible notebook for the analysis of `milei_corpus_v4_code_counts.csv` (193 coded speeches, Dimension 1 codebook v4).

**What this notebook does, in order:**
1. Data prep & QA — load the corpus, compute word counts, build a long-format code table
2. Descriptive overview — code distributions, sub-code ranking, zero-code speech profile
3. Time trend — code density by quarter across the presidency
4. Venue & country effects — cross-tabs by `tipo`/`pais` with Kruskal-Wallis tests
5. Climate-focused deep dive — validating the `clima` heuristic, "climate-signature" sub-codes, top speeches
6. Co-occurrence & rhetorical-mode clustering — sub-code correlation matrix + k-means clusters

**Requirements:** `pandas`, `numpy`, `scipy`, `scikit-learn`, `matplotlib`. Install with:
```
pip install pandas numpy scipy scikit-learn matplotlib
```

**Paths:** set `BASE_DIR` below to the root of the `milei-climate-narratives` project folder (the folder containing `analysis/` and `data/`). Everything else is relative to it.

In [1]:
import os
import re
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib
matplotlib.use('Agg')  # switch to default backend if running interactively, e.g. remove this line in Jupyter
import matplotlib.pyplot as plt

# --- EDIT THIS to point at your local project folder ---
BASE_DIR = "."  # e.g. r"C:\Users\you\milei-climate-narratives"

CORPUS_CSV = os.path.join(BASE_DIR, "analysis/results/v4/milei_corpus_v4_code_counts.csv")
RAW_SPEECHES_DIR = os.path.join(BASE_DIR, "data/raw_speeches")
OUT_DIR = os.path.join(BASE_DIR, "analysis/results/v4/quant_analysis")
CHARTS_DIR = os.path.join(OUT_DIR, "charts")
os.makedirs(CHARTS_DIR, exist_ok=True)

pd.set_option('display.width', 140)
pd.set_option('display.max_rows', 40)

## 1. Data prep & QA

Load the master code-count CSV, confirm all rows are `coded=ok`, and compute a **word count per speech** from the raw (uncoded) transcripts in `data/raw_speeches/` — using the raw files rather than the annotated ones in `analysis/coded_speeches/v4/` avoids inflating counts with `[CODE: ...]` tags. Word counts let us normalize code counts by speech length later (a 300-word cadena nacional clip and a 7,700-word Davos keynote are not comparable on raw counts alone).

In [2]:
df = pd.read_csv(CORPUS_CSV)
print("Rows:", len(df), "| Columns:", len(df.columns))
print("coded == 'ok' for all rows:", (df['coded'] == 'ok').all())

subcodes = [
    'D1.1_Market_Primacy', 'D1.1_Anti_State_Anti_Bureaucratic', 'D1.1_Anti_Casta',
    'D1.1_Delegitimization_of_Regulation', 'D1.1_Sustainability_as_Ideological_or_Harmful',
    'D1.1_Rejection_of_Multilateral_Governance',
    'D1.2_Attack_on_Universities_and_Research_Institutions', 'D1.2_Attack_on_Experts_and_Technocrats',
    'D1.2_Climate_Skepticism_or_Dismissal', 'D1.2_Pseudo_Science_or_Ideological_Science', 'D1.2_Selective_Empiricism',
    'D1.3_Woke_Bundling', 'D1.3_Anti_Socialism_Anti_Collectivism', 'D1.3_Civilizational_Defense',
]

Rows: 193 | Columns: 32
coded == 'ok' for all rows: True


In [3]:
def word_count(fname):
    """Word count from the raw (unannotated) transcript matched via `matched_coded_file`."""
    path = os.path.join(RAW_SPEECHES_DIR, fname)
    if not os.path.exists(path):
        return None
    txt = open(path, encoding='utf-8').read()
    # strip YAML frontmatter (content between the first two '---' markers)
    parts = txt.split('---')
    body = '---'.join(parts[2:]) if len(parts) >= 3 else txt
    return len(re.findall(r"\b\w+\b", body, re.UNICODE))

df['word_count'] = df['matched_coded_file'].apply(word_count)
print("Missing word counts:", df['word_count'].isna().sum())
df['word_count'].describe()

Missing word counts: 0


count      193.000000
mean      3963.435233
std       3112.700560
min        152.000000
25%       1238.000000
50%       3174.000000
75%       6419.000000
max      11702.000000
Name: word_count, dtype: float64

In [4]:
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
print("Date parse failures:", df['fecha'].isna().sum())
print("Date range:", df['fecha'].min().date(), "to", df['fecha'].max().date())

df['year'] = df['fecha'].dt.year
df['quarter'] = df['fecha'].dt.to_period('Q').astype(str)
df['month'] = df['fecha'].dt.to_period('M').astype(str)

# Rates per 1,000 words -- the primary normalized metric used throughout this notebook
df['codes_per_1k'] = df['Codes_Total'] / df['word_count'] * 1000
df['d11_per_1k'] = df['D1.1_Total'] / df['word_count'] * 1000
df['d12_per_1k'] = df['D1.2_Total'] / df['word_count'] * 1000
df['d13_per_1k'] = df['D1.3_Total'] / df['word_count'] * 1000

df.to_csv(os.path.join(OUT_DIR, 'corpus_enriched.csv'), index=False)
df[['fecha', 'quarter', 'word_count', 'Codes_Total', 'codes_per_1k']].head()

Date parse failures: 0
Date range: 2023-10-08 to 2026-04-29


,fecha,quarter,word_count,Codes_Total,codes_per_1k
0,2023-10-08,2023Q4,312,3,9.615385
1,2023-12-10,2023Q4,3446,5,1.450958
2,2023-12-10,2023Q4,354,0,0.000000
3,2023-12-20,2023Q4,2268,7,3.086420
4,2023-12-30,2023Q4,1150,3,2.608696


Build a **long-format table** (one row per code instance) alongside the wide table — this makes sub-code ranking and co-occurrence analysis much simpler than pivoting the wide table each time.

In [5]:
long = df.melt(
    id_vars=['id', 'fecha', 'year', 'quarter', 'pais', 'tipo', 'subtipo', 'clima', 'word_count'],
    value_vars=subcodes, var_name='subcode', value_name='count'
)
long = long[long['count'] > 0].copy()
long['dimension'] = long['subcode'].str.split('_').str[0]
long.to_csv(os.path.join(OUT_DIR, 'long_format.csv'), index=False)
print("Long-format rows (code instances):", len(long))

Long-format rows (code instances): 451


## 2. Descriptive overview

Distribution of total codes per speech, ranking of the 14 sub-codes by frequency, and a profile of the zero-code speeches (do they cluster in a particular genre, or are they scattered randomly?).

In [6]:
print(df['Codes_Total'].describe())
print()
dim_totals = df[['D1.1_Total', 'D1.2_Total', 'D1.3_Total']].sum()
print("Dimension totals:")
print(dim_totals)
print((dim_totals / dim_totals.sum() * 100).round(1), "% share")

count    193.000000
mean       2.787565
std        2.600371
min        0.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       19.000000
Name: Codes_Total, dtype: float64

Dimension totals:
D1.1_Total    329
D1.2_Total     63
D1.3_Total    146
dtype: int64
D1.1_Total    61.2
D1.2_Total    11.7
D1.3_Total    27.1
dtype: float64 % share


In [7]:
sub_totals = long.groupby('subcode')['count'].sum().sort_values(ascending=False)
sub_totals

subcode
D1.1_Anti_Casta                                          99
D1.3_Anti_Socialism_Anti_Collectivism                    86
D1.1_Anti_State_Anti_Bureaucratic                        84
D1.1_Delegitimization_of_Regulation                      69
D1.3_Woke_Bundling                                       41
D1.2_Attack_on_Experts_and_Technocrats                   31
D1.1_Market_Primacy                                      29
D1.1_Rejection_of_Multilateral_Governance                24
D1.1_Sustainability_as_Ideological_or_Harmful            24
D1.3_Civilizational_Defense                              19
D1.2_Attack_on_Universities_and_Research_Institutions    14
D1.2_Pseudo_Science_or_Ideological_Science                8
D1.2_Climate_Skepticism_or_Dismissal                      7
D1.2_Selective_Empiricism                                 3
Name: count, dtype: int64

In [8]:
zero = df[df['Codes_Total'] == 0]
print("Zero-code speeches:", len(zero), f"({len(zero)/len(df)*100:.0f}% of corpus)")
print()
print("By tipo:")
print(zero['tipo'].value_counts())
print()
print("By pais:")
print(zero['pais'].value_counts())
print()
print("By clima flag:")
print(zero['clima'].fillna('blank').value_counts())

Zero-code speeches: 33 (17% of corpus)

By tipo:
tipo
acto institucional                7
acto conmemorativo                5
bilateral / diplomatico           5
cumbre internacional              3
evento ideologico/politico        3
foro economico                    2
premio / reconocimiento           2
sectorial                         1
clase magistral / universidad     1
acto institucional / bilateral    1
evento economico / cultural       1
comunitario / religioso           1
acto conmemorativo/patrio         1
Name: count, dtype: int64

By pais:
pais
argentina      17
israel          6
ee.uu.          5
el salvador     1
suiza           1
italia          1
paraguay        1
españa          1
Name: count, dtype: int64

By clima flag:
clima
blank    24
~         9
Name: count, dtype: int64


In [9]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df['Codes_Total'], bins=range(0, 21), color='#4A4A4A', edgecolor='white')
ax.set_xlabel('Codes_Total per speech')
ax.set_ylabel('Number of speeches')
ax.set_title(f"Distribution of D1 code counts per speech (n={len(df)}, {(df.Codes_Total==0).sum()} zero-code)")
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '03_codes_histogram.png'), dpi=150)
plt.show()

In [10]:
short_names = [c.split('_', 1)[1].replace('_', ' ') for c in sub_totals.index]
colors = ['#8B2E2E' if s.startswith('D1.1') else '#2E6B4A' if s.startswith('D1.2') else '#2E4A8B'
          for s in sub_totals.index]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(short_names[::-1], sub_totals.values[::-1], color=colors[::-1])
ax.set_xlabel('Total instances across corpus')
ax.set_title('D1 sub-code frequency ranking\n(red=D1.1 State/Sustainability, green=D1.2 Science/Expertise, blue=D1.3 Cultural Backlash)')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '02_subcode_ranking.png'), dpi=150)
plt.show()

## 3. Time trend

Aggregate by quarter and compute **code density** (codes per 1,000 words), not just raw code volume, since the number and length of speeches per quarter varies a lot.

In [11]:
q = df.groupby('quarter').agg(
    n_speeches=('id', 'count'),
    total_codes=('Codes_Total', 'sum'),
    total_words=('word_count', 'sum'),
).reset_index()
q['codes_per_1k'] = q['total_codes'] / q['total_words'] * 1000
q

,quarter,n_speeches,total_codes,total_words,codes_per_1k
0,2023Q4,5,18,7530,2.390438
1,2024Q1,7,34,25917,1.311880
2,2024Q2,27,68,114024,0.596366
3,2024Q3,20,90,89593,1.004543
4,2024Q4,34,90,132734,0.678048
5,2025Q1,10,38,38595,0.984583
6,2025Q2,20,32,84123,0.380395
7,2025Q3,25,75,92497,0.810837
8,2025Q4,16,43,51957,0.827607
9,2026Q1,20,39,88224,0.442057


In [12]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].bar(q['quarter'], q['total_codes'], color='#8B2E2E')
axes[0].set_ylabel('Total D1 codes')
axes[0].set_title('D1 code volume by quarter (raw count)')
axes[0].tick_params(axis='x', rotation=90)

axes[1].plot(q['quarter'], q['codes_per_1k'], marker='o', color='#2E4A8B')
axes[1].set_ylabel('Codes per 1,000 words')
axes[1].set_title('D1 code density by quarter (normalized by word count)')
axes[1].tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '01_time_series.png'), dpi=150)
plt.show()

## 4. Venue & country effects

Cross-tabulate code density (`codes_per_1k`) by `tipo` (venue/speech type) and `pais` (country), and test for group differences with the **Kruskal-Wallis** test (nonparametric — the rate distribution is right-skewed, so ANOVA's normality assumption doesn't hold). Groups below a minimum size are excluded from the test itself since a single-speech "group" doesn't provide a meaningful test of central tendency.

In [13]:
tipo_stats = df.groupby('tipo').agg(
    n=('id', 'count'),
    mean_codes_per_1k=('codes_per_1k', 'mean'),
    median_codes_per_1k=('codes_per_1k', 'median'),
    mean_word_count=('word_count', 'mean'),
).sort_values('mean_codes_per_1k', ascending=False)
tipo_stats.to_csv(os.path.join(OUT_DIR, 'tipo_stats.csv'))
tipo_stats.round(2)

,n,mean_codes_per_1k,median_codes_per_1k,mean_word_count
tipo,,,,
debate presidencial,1,9.62,9.62,312.00
cumbre internacional,12,2.90,1.57,1083.00
evento empresarial,1,1.72,1.72,2913.00
empresa,10,1.36,1.18,2560.50
cadena nacional,10,1.29,0.99,2980.10
foro economico / financiero,1,1.25,1.25,2404.00
acto conmemorativo/patrio,6,1.14,1.14,1001.50
bilateral / diplomatico,13,1.10,0.86,1043.38
evento ideologico/politico,21,1.10,0.99,4392.43


In [14]:
# Kruskal-Wallis across tipo groups with n >= 5 (stable estimate threshold)
groups_tipo = [g['codes_per_1k'].values for name, g in df.groupby('tipo') if len(g) >= 5]
h, p = stats.kruskal(*groups_tipo)
print(f"Kruskal-Wallis across tipo (k={len(groups_tipo)} groups with n>=5): H={h:.2f}, p={p:.4f}")

Kruskal-Wallis across tipo (k=15 groups with n>=5): H=32.38, p=0.0035


In [15]:
pais_stats = df.groupby('pais').agg(
    n=('id', 'count'),
    mean_codes_per_1k=('codes_per_1k', 'mean'),
    median_codes_per_1k=('codes_per_1k', 'median'),
).sort_values('mean_codes_per_1k', ascending=False)
pais_stats.to_csv(os.path.join(OUT_DIR, 'pais_stats.csv'))
pais_stats.round(2)

,n,mean_codes_per_1k,median_codes_per_1k
pais,,,
brasil,4,3.22,3.11
suiza,5,1.75,0.90
ee.uu.,21,1.64,0.73
paraguay,5,1.50,0.58
hungria,2,1.48,1.48
uruguay,3,1.22,0.97
españa,8,1.00,0.53
argentina,129,0.86,0.62
chile,1,0.60,0.60


In [16]:
groups_pais = [g['codes_per_1k'].values for name, g in df.groupby('pais') if len(g) >= 3]
h2, p2 = stats.kruskal(*groups_pais)
print(f"Kruskal-Wallis across pais (k={len(groups_pais)} groups with n>=3): H={h2:.2f}, p={p2:.4f}")

# Argentina vs. rest-of-world binary comparison
df['ar_vs_intl'] = np.where(df['pais'] == 'argentina', 'argentina', 'international')
u, pu = stats.mannwhitneyu(
    df.loc[df.ar_vs_intl == 'argentina', 'codes_per_1k'],
    df.loc[df.ar_vs_intl == 'international', 'codes_per_1k'],
)
print(df.groupby('ar_vs_intl')['codes_per_1k'].agg(['count', 'mean', 'median']).round(3))
print(f"Mann-Whitney U={u:.1f}, p={pu:.4f}")

Kruskal-Wallis across pais (k=8 groups with n>=3): H=14.33, p=0.0456
               count   mean  median
ar_vs_intl                         
argentina        129  0.855   0.618
international     64  1.309   0.697
Mann-Whitney U=4043.0, p=0.8166


## 5. Climate-focused deep dive

Two questions here:

1. **Does the corpus's `clima` selection heuristic (✔ / ~ / blank) actually predict code density?** If yes, that validates using it as a speech-selection filter.
2. **Does climate-obstruction rhetoric bleed into speeches *not* flagged as climate-relevant?** Defined via three "climate-signature" sub-codes: `Sustainability_as_Ideological_or_Harmful`, `Rejection_of_Multilateral_Governance`, `Climate_Skepticism_or_Dismissal`. If a large share of these instances falls outside `clima=✔` speeches, that supports coding climate-adjacent rhetoric wherever it appears rather than gating on an explicit climate mention per passage.

In [17]:
df['clima_f'] = df['clima'].fillna('blank')

profile = df.groupby('clima_f').agg(
    n=('id', 'count'),
    mean_codes_per_1k=('codes_per_1k', 'mean'),
    mean_d11=('d11_per_1k', 'mean'),
    mean_d12=('d12_per_1k', 'mean'),
    mean_d13=('d13_per_1k', 'mean'),
)
print(profile.round(3))

groups_clima = [g['codes_per_1k'].values for name, g in df.groupby('clima_f')]
h3, p3 = stats.kruskal(*groups_clima)
print(f"\nKruskal-Wallis across clima groups: H={h3:.2f}, p={p3:.4f}")

           n  mean_codes_per_1k  mean_d11  mean_d12  mean_d13
clima_f                                                      
blank     72              0.904     0.589     0.057     0.258
~        111              0.728     0.427     0.060     0.241
✔         10              4.824     3.117     1.037     0.670

Kruskal-Wallis across clima groups: H=17.92, p=0.0001


In [18]:
fig, ax = plt.subplots(figsize=(7, 5))
order = ['blank', '~', '✔']
vals = [df.loc[df.clima_f == c, 'codes_per_1k'].mean() for c in order]
labels = ['No climate\nrelevance flagged', 'Possible climate\nrelevance (~)', 'High climate\nrelevance (✔)']
ax.bar(labels, vals, color=['#999999', '#D9A441', '#8B2E2E'])
ax.set_ylabel('Mean D1 codes per 1,000 words')
ax.set_title(f'D1 code density by climate-relevance heuristic (Kruskal-Wallis p={p3:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '04_clima_density.png'), dpi=150)
plt.show()

In [19]:
sig_cols = [
    'D1.1_Sustainability_as_Ideological_or_Harmful',
    'D1.1_Rejection_of_Multilateral_Governance',
    'D1.2_Climate_Skepticism_or_Dismissal',
]
df['climate_signature_total'] = df[sig_cols].sum(axis=1)
df['climate_signature_per_1k'] = df['climate_signature_total'] / df['word_count'] * 1000

inside = df.loc[df.clima_f == '✔', 'climate_signature_total'].sum()
outside = df.loc[df.clima_f != '✔', 'climate_signature_total'].sum()
total = inside + outside
print(f"Climate-signature instances inside clima=✔ speeches: {inside} ({inside/total*100:.0f}%)")
print(f"Climate-signature instances outside clima=✔ speeches: {outside} ({outside/total*100:.0f}%)")
print(f"Total: {total}")

df.to_csv(os.path.join(OUT_DIR, 'corpus_enriched.csv'), index=False)  # re-save with the new columns

Climate-signature instances inside clima=✔ speeches: 23 (42%)
Climate-signature instances outside clima=✔ speeches: 32 (58%)
Total: 55


In [20]:
top = (
    df[df['climate_signature_total'] > 0]
    .sort_values('climate_signature_per_1k', ascending=False)
    [['fecha', 'evento-discurso', 'pais', 'tipo', 'clima_f', 'climate_signature_total', 'word_count', 'climate_signature_per_1k']]
    .head(15)
)
top

,fecha,evento-discurso,pais,tipo,clima_f,climate_signature_total,word_count,climate_signature_per_1k
58,2024-09-24,79 asamblea general de la onu – debate general,ee.uu.,cumbre internacional,✔,8,1067,7.497657
0,2023-10-08,debate presidencial 2023,argentina,debate presidencial,✔,2,312,6.410256
144,2025-09-24,80 asamblea general de la onu,ee.uu.,cumbre internacional,✔,5,1654,3.022975
75,2024-11-18,cumbre de lideres del g20 – sesion ii,brasil,cumbre internacional,✔,2,770,2.597403
74,2024-11-18,cumbre de lideres del g20 – intervencion,brasil,cumbre internacional,✔,1,876,1.141553
92,2024-12-21,anuncios en materia nuclear (plan nuclear arge...,argentina,acto institucional,✔,1,1238,0.807754
180,2026-03-21,cpac budapest,hungria,evento ideologico/politico,~,3,3742,0.801710
117,2025-06-12,memorando de la democracia y la libertad (con ...,israel,bilateral / diplomatico,blank,1,1334,0.749625
46,2024-07-28,"exposicion de ganaderia, agricultura e industr...",argentina,sectorial,~,3,4299,0.697837
96,2025-01-23,foro economico mundial de davos 2025,suiza,foro economico,✔,3,4456,0.673250


In [21]:
q_sig = df.groupby('quarter').agg(sig_sum=('climate_signature_total', 'sum')).reset_index()
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(q_sig['quarter'], q_sig['sig_sum'], color='#2E6B4A')
ax.set_ylabel('Climate-signature code instances')
ax.set_title('Climate-signature codes (Sustainability_as_Ideological, Rejection_of_Multilateral_Gov, Climate_Skepticism) by quarter')
ax.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '05_climate_signature_time.png'), dpi=150)
plt.show()

## 6. Co-occurrence and rhetorical-mode clustering

**Co-occurrence:** phi coefficients (Pearson correlation on binary presence/absence) across the 14 sub-codes — which codes tend to appear together within the same speech?

**Clustering:** k-means on each speech's code-**composition** profile (i.e. what % of that speech's codes belong to each sub-code — the "shape" of its rhetorical mix, independent of how many codes it has overall). Composition is used instead of raw rate-per-word because very short, high-rate outlier speeches (e.g. a 300-word debate clip) otherwise dominate a rate-based clustering. Only speeches with `Codes_Total >= 3` are included so each profile is estimated from enough data points to be stable.

In [22]:
X_bin = (df[subcodes] > 0).astype(int)
corr = X_bin.corr()

fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(short_names))); ax.set_xticklabels(short_names, rotation=90)
ax.set_yticks(range(len(short_names))); ax.set_yticklabels(short_names)
plt.colorbar(im, label='Phi coefficient (co-occurrence correlation)')
ax.set_title(f'Sub-code co-occurrence correlation (presence/absence per speech, n={len(df)})')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '06_cooccurrence_heatmap.png'), dpi=150)
plt.show()

In [23]:
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
print("Top 10 co-occurring sub-code pairs:")
pairs.head(10)

Top 10 co-occurring sub-code pairs:


D1.2_Pseudo_Science_or_Ideological_Science             D1.2_Selective_Empiricism                        0.423692
D1.1_Rejection_of_Multilateral_Governance              D1.3_Woke_Bundling                               0.402011
                                                       D1.3_Civilizational_Defense                      0.269249
D1.2_Attack_on_Universities_and_Research_Institutions  D1.2_Climate_Skepticism_or_Dismissal             0.266286
D1.3_Woke_Bundling                                     D1.3_Civilizational_Defense                      0.258090
D1.2_Attack_on_Experts_and_Technocrats                 D1.2_Pseudo_Science_or_Ideological_Science       0.248143
D1.1_Sustainability_as_Ideological_or_Harmful          D1.2_Attack_on_Experts_and_Technocrats           0.238759
D1.1_Delegitimization_of_Regulation                    D1.1_Sustainability_as_Ideological_or_Harmful    0.236710
D1.1_Anti_State_Anti_Bureaucratic                      D1.1_Anti_Casta                          

In [24]:
sub = df[df['Codes_Total'] >= 3].copy()
comp = sub[subcodes].div(sub['Codes_Total'], axis=0)  # composition: each subcode's share of that speech's total codes
X = StandardScaler().fit_transform(comp)

print(f"n speeches with Codes_Total>=3: {len(sub)}\n")
for k in range(2, 7):
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    score = silhouette_score(X, km_k.labels_)
    print(f"k={k}: silhouette={score:.3f}, cluster sizes={np.bincount(km_k.labels_)}")

n speeches with Codes_Total>=3: 86

k=2: silhouette=0.126, cluster sizes=[21 65]
k=3: silhouette=0.151, cluster sizes=[55 17 14]
k=4: silhouette=0.130, cluster sizes=[ 7 27 10 42]
k=5: silhouette=0.105, cluster sizes=[39  9 19 18  1]
k=6: silhouette=0.155, cluster sizes=[16 39  2 17 10  2]


In [25]:
# k=4 chosen: interpretable clusters at a reasonable silhouette score for compositional political-rhetoric data
K = 4
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X)
sub['cluster'] = km.labels_

for c in sorted(sub['cluster'].unique()):
    grp = sub[sub['cluster'] == c]
    print(f"--- Cluster {c} (n={len(grp)}, mean codes/1k={grp['codes_per_1k'].mean():.2f}) ---")
    top_codes = comp.loc[grp.index].mean().sort_values(ascending=False).head(4)
    print((top_codes * 100).round(1).astype(str) + '%')
    print('tipo top3:', grp['tipo'].value_counts().head(3).to_dict())
    print('pais top3:', grp['pais'].value_counts().head(3).to_dict())
    print()

sub.to_csv(os.path.join(OUT_DIR, 'clustered_speeches.csv'), index=False)

--- Cluster 0 (n=7, mean codes/1k=4.11) ---
D1.1_Rejection_of_Multilateral_Governance    46.0%
D1.3_Woke_Bundling                           18.2%
D1.1_Delegitimization_of_Regulation          14.5%
D1.1_Anti_State_Anti_Bureaucratic             9.0%
dtype: object
tipo top3: {'cumbre internacional': 4, 'bolsa / camara de comercio': 1, 'bilateral / diplomatico': 1}
pais top3: {'ee.uu.': 3, 'argentina': 2, 'brasil': 1}

--- Cluster 1 (n=27, mean codes/1k=1.03) ---
D1.1_Anti_Casta                          27.6%
D1.3_Anti_Socialism_Anti_Collectivism    27.3%
D1.1_Anti_State_Anti_Bureaucratic        12.1%
D1.3_Woke_Bundling                       11.9%
dtype: object
tipo top3: {'evento ideologico/politico': 7, 'foro economico': 6, 'bolsa / camara de comercio': 3}
pais top3: {'argentina': 15, 'españa': 4, 'ee.uu.': 3}

--- Cluster 2 (n=10, mean codes/1k=1.21) ---
D1.2_Attack_on_Universities_and_Research_Institutions    20.1%
D1.1_Anti_Casta                                          19.5%
D1.1_Del

In [26]:
pca = PCA(n_components=2).fit(X)
coords = pca.transform(X)
sub['pc1'], sub['pc2'] = coords[:, 0], coords[:, 1]

cluster_labels = {0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2', 3: 'Cluster 3'}  # relabel descriptively after inspecting means above
colors = {0: '#8B2E2E', 1: '#2E4A8B', 2: '#2E6B4A', 3: '#999999'}

fig, ax = plt.subplots(figsize=(9, 7))
for c in sorted(sub['cluster'].unique()):
    grp = sub[sub['cluster'] == c]
    ax.scatter(grp['pc1'], grp['pc2'], label=f"{cluster_labels[c]} (n={len(grp)})", color=colors[c], s=60, alpha=0.8)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% var)')
ax.set_title(f'Rhetorical-mode clusters by D1 code composition (Codes_Total>=3, n={len(sub)})')
ax.legend(fontsize=8, loc='best')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '07_cluster_pca.png'), dpi=150)
plt.show()

## Done

All intermediate tables (`corpus_enriched.csv`, `long_format.csv`, `tipo_stats.csv`, `pais_stats.csv`, `clustered_speeches.csv`) and all 7 charts are written to `analysis/results/v4/quant_analysis/`. See `quant_analysis_memo.md` in the same folder for the full write-up of findings and their implications for the paper draft.